# Exploratory Data Analysis (EDA) - AQI India
This notebook explores the air quality data, showing the steps taken to understand the distributions, correlations, and time-series patterns before building the dashboard and forecasting models.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('ggplot')
sns.set_palette('muted')


ModuleNotFoundError: No module named 'seaborn'

## 1. Data Loading
In the actual application, data is fetched live from the OpenAQ API. For the purpose of EDA, let's load a sample dataset of historical measurements. If historical data isn't available locally, we will simulate loading and processing the data structures returned by our `fetch_data.py` script.


In [2]:
import os
import sys

# Ensure we can import from our project
sys.path.append(os.path.abspath("."))
from data.fetch_data import fetch_latest_india

print("Fetching latest AQI data from OpenAQ API for EDA...")
df = fetch_latest_india()
display(df.head())


2026-06-17 17:23:41.945 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-17 17:23:41.946 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-17 17:23:41.947 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-17 17:23:41.947 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Fetching latest AQI data from OpenAQ API for EDA...


,city,lat,lon,pm25,pm10,no2,co,so2,o3,last_updated
0,Delhi,28.599998,77.200010,103.8,560.7,7.7,259.0,16.1,165.0,2026-06-17T11:00
1,Mumbai,19.099998,72.900010,12.0,22.5,5.9,164.0,5.4,68.0,2026-06-17T11:00
2,Bengaluru,13.000000,77.600006,27.9,56.8,10.1,542.0,3.8,96.0,2026-06-17T11:00
3,Chennai,13.099998,80.300020,27.8,43.5,4.7,295.0,6.8,145.0,2026-06-17T11:00
4,Kolkata,22.599998,88.399994,43.7,59.0,6.3,417.0,8.4,118.0,2026-06-17T11:00


## 2. Data Cleaning & Preprocessing
Let's check for missing values and understand the dataset's structure.


In [ ]:
# Check data types and missing values
print(df.info())

print("\nMissing values per column:")
print(df.isnull().sum())

# We might have different pollutants in the 'parameter' column. Let's see them:
print("\nAvailable parameters:")
print(df['parameter'].value_counts())


To analyze correlations, we should pivot the dataset so that each pollutant has its own column.


In [ ]:
# Pivot the data
if not df.empty:
    df_pivot = df.pivot_table(index=['city', 'location', 'last_updated'], 
                              columns='parameter', 
                              values='value').reset_index()
    display(df_pivot.head())


## 3. Distributions and Outliers
Let's look at the distribution of PM2.5 and PM10 across the fetched cities. We use boxplots to identify extreme pollution events (outliers).


In [ ]:
if not df.empty and 'pm25' in df_pivot.columns and 'pm10' in df_pivot.columns:
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    sns.boxplot(y=df_pivot['pm25'])
    plt.title('Distribution of PM2.5')
    plt.ylabel('Concentration (µg/m³)')
    
    plt.subplot(1, 2, 2)
    sns.boxplot(y=df_pivot['pm10'])
    plt.title('Distribution of PM10')
    plt.ylabel('Concentration (µg/m³)')
    
    plt.tight_layout()
    plt.show()


## 4. Correlation Matrix
How do different pollutants correlate with each other? For instance, does PM2.5 rise with CO or NO2?


In [ ]:
if not df.empty:
    # Select numerical columns (pollutants)
    pollutants = df_pivot.select_dtypes(include=[np.number])
    
    if not pollutants.empty and pollutants.shape[1] > 1:
        corr = pollutants.corr()
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
        plt.title('Correlation Matrix of Pollutants')
        plt.show()
    else:
        print("Not enough pollutant data to generate a correlation matrix.")


## 5. Time-Series Analysis Justification
The Prophet model expects a dataframe with `ds` (datetime) and `y` (target variable).
Let's see how a typical time-series would look. (Using simulated historical variation based on current values for demonstration if actual historical endpoints aren't queried here).


In [ ]:
if not df.empty and 'pm25' in df_pivot.columns:
    # Let's take the city with the highest PM2.5 currently
    worst_city = df_pivot.loc[df_pivot['pm25'].idxmax()]['city']
    print(f"Analyzing time-series structure for: {worst_city}")
    
    # Simulate 30 days of past data around the current value for EDA visualization
    # In production, this data is fetched from the OpenAQ /measurements endpoint
    current_val = df_pivot['pm25'].max()
    dates = pd.date_range(end=pd.Timestamp.today(), periods=30)
    
    # Add some trend, weekly seasonality and noise
    trend = np.linspace(-20, 0, 30)
    seasonality = np.sin(np.arange(30) * (2 * np.pi / 7)) * 15
    noise = np.random.normal(0, 10, 30)
    
    simulated_pm25 = np.maximum(current_val + trend + seasonality + noise, 0)
    
    ts_df = pd.DataFrame({'ds': dates, 'y': simulated_pm25})
    
    plt.figure(figsize=(12, 5))
    plt.plot(ts_df['ds'], ts_df['y'], marker='o', linestyle='-')
    plt.title(f'Simulated 30-Day PM2.5 Trend for {worst_city}')
    plt.xlabel('Date')
    plt.ylabel('PM2.5 (µg/m³)')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("This temporal structure (trend + seasonality + noise) is exactly what Meta's Prophet model is designed to decompose and forecast.")
